# RadioSSL Kaggle runbook (P100)

Run the repository version containing the patient-aware split and separate NPZ packaging. Set the repository URL and Kaggle input path below. Attach `ILD_DB.zip` as a Kaggle Dataset input instead of placing it in Git. The source audit reports the actual folder IDs; stride-32 extraction writes a training NPZ and a separate real-patient Talisman validation NPZ. All volume and HRCT pilot cases are retained for training. Exact numeric IDs shared with the training cohort are removed from Talisman validation to keep validation patient-disjoint. The password-protected clinical workbook is not copied into code or opened by the notebook.


In [ ]:
# Kaggle notebook settings
REPO_URL = "https://github.com/Siddharthakhandelwal/o1.git"
DATASET_ZIP = "/kaggle/input/your-ild-dataset/ILD_DB.zip"
REPO_DIR = "/kaggle/working/radiossl"
DATA_DIR = "/kaggle/working/data"
OUTPUT_DIR = "/kaggle/working/radiossl_run"

import os, torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB VRAM')
assert os.path.isfile(DATASET_ZIP), f'Dataset archive not found: {DATASET_ZIP}'

## 1. Clone the repository

In [ ]:
import subprocess
subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd /kaggle/working/radiossl
!git rev-parse HEAD

## 2. Install dependencies (keep Kaggle CUDA PyTorch)

In [ ]:
# requirements-kaggle.txt should list project libraries, excluding torch/torchvision.
!python -m pip install -r requirements-kaggle.txt
!python -c "import torch, numpy, pydicom, tifffile, scipy, sklearn, skimage; print('imports ok; torch', torch.__version__, 'CUDA', torch.version.cuda)"

## 3. Extract and audit the original source data

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
!unzip -q -n "{DATASET_ZIP}" -d "{DATA_DIR}"
!python scripts/audit_source_dataset.py --base_dir "{DATA_DIR}/ILD_DB" --output "{OUTPUT_DIR}/source_audit.json"

In [ ]:
import json
audit = json.load(open(f'{OUTPUT_DIR}/source_audit.json', encoding='utf-8'))
print({k: audit[k] for k in ['talisman_real_patient_id_count','volume_folder_id_count','hrct_pilot_case_count','volume_and_pilot_identifier_count','numeric_id_overlap_count','clinical_workbook_status','status']})
print('All volume/pilot cases train; matching Talisman IDs are excluded from validation.')

## 4. Build stride-32 patches and preserve labels/provenance

In [ ]:
!python step2_extract_and_package.py --base_dir "{DATA_DIR}/ILD_DB" --output_dir "{DATA_DIR}/processed" --stride 32
!python scripts/make_patient_split.py --data_path "{DATA_DIR}/processed/ild_train_dataset.npz" --test_size 0 --seed 42 --patient_ids_same_namespace --output "{DATA_DIR}/patient_split.json"

## 5. Preflight: split checks and a few GPU steps
Confirm patient-disjoint assertions pass and the short run reports finite loss/gradients and peak VRAM before starting a long job.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
!python train_radiossl.py --data_path "{DATA_DIR}/processed/ild_train_dataset.npz" --split_manifest "{DATA_DIR}/patient_split.json" --output_dir "{OUTPUT_DIR}/preflight" --epochs 1 --preflight_steps 3 --batch_size 16 --accum_steps 1 --amp

## 6. One full patient-aware P100 pretraining run
Do not run until the preflight passes. Test patients must not enter SSL, SupCon, radiomics scaling or checkpoint selection. Epoch count is fixed in advance; final held-out evaluation runs once.

In [ ]:
!python train_radiossl.py --data_path "{DATA_DIR}/processed/ild_train_dataset.npz" --split_manifest "{DATA_DIR}/patient_split.json" --output_dir "{OUTPUT_DIR}/pretrain" --epochs 350 --batch_size 128 --accum_steps 2 --amp

## 7. Final probe, optional fine-tuning, and save artifacts

In [ ]:
!python scripts/evaluate_linear_probe.py --data_path "{DATA_DIR}/processed/ild_train_dataset.npz" --split_manifest "{DATA_DIR}/patient_split.json" --validation_data_path "{DATA_DIR}/processed/ild_talisman_validation.npz" --checkpoint "{OUTPUT_DIR}/pretrain/radiossl_encoder_final.pth" --output "{OUTPUT_DIR}/linear_probe.json"
# The optional fine-tuning script currently expects an internal train/test manifest.
# This runbook uses external Talisman validation with all eligible volume patients in SSL,
# so do not execute fine-tune until it supports the same external-validation argument.
!cd /kaggle/working && zip -qr radiossl_run_artifacts.zip radiossl_run data/patient_split.json data/processed/dataset_summary.json